# Task 6 — Nationality Detection Model
**Internship ML Project | Google Colab + GPU**

### Pipeline Overview
| Nationality | Models Applied |
|---|---|
| 🇮🇳 Indian | Nationality + Emotion + Age + Dress Colour |
| 🇺🇸 American | Nationality + Emotion + Age |
| 🌍 African | Nationality + Emotion + Dress Colour |
| 🌐 Others | Nationality + Emotion only |

### Models (All trained from scratch)
1. **Nationality CNN** — UTKFace dataset (4-block CNN)
2. **Emotion CNN** — FER2013 dataset (4-block CNN)
3. **Age CNN** — UTKFace dataset (4-block CNN)
4. **Dress Colour CNN** — Human Parsing / Color dataset (3-block CNN)

**Version:** v3 | Fully self-contained | Kaggle API for all datasets

## Cell 1 — Install Dependencies

In [ ]:
# Cell 1 — Install packages
# Run once. No runtime restart needed.

!pip install -q kaggle opencv-python-headless matplotlib seaborn scikit-learn
print('✅ Packages installed.')


## Cell 2 — Imports

In [ ]:
# Cell 2 — Imports

# ── Protobuf compatibility shim ───────────────────────────────────────────────
# TF 2.16+ calls runtime_version.ValidateProtobufRuntimeVersion(Domain.PUBLIC, ...)
# We inject a complete fake module that satisfies all attribute lookups.
import sys, types
import google.protobuf

if not hasattr(google.protobuf, 'runtime_version') or \
   not hasattr(google.protobuf.runtime_version, 'Domain'):

    rv = types.ModuleType('google.protobuf.runtime_version')

    # Domain must be an object (enum-like) with a PUBLIC attribute
    class _Domain:
        PUBLIC = 'PUBLIC'
        GOOGLE_INTERNAL = 'GOOGLE_INTERNAL'
    rv.Domain = _Domain

    # ValidateProtobufRuntimeVersion is called with Domain.PUBLIC + version ints
    rv.ValidateProtobufRuntimeVersion = lambda *a, **kw: None

    rv.OSS = True
    rv.VersionInfo = (5, 29, 6, 0, '', 0)

    sys.modules['google.protobuf.runtime_version'] = rv
    google.protobuf.runtime_version = rv
    print(f'✅ protobuf shim applied (version {google.protobuf.__version__})')
else:
    print(f'✅ protobuf {google.protobuf.__version__} — no shim needed')

# ── Imports ───────────────────────────────────────────────────────────────────
import os, shutil, random, zipfile, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from google.colab import drive, files

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow version : {tf.__version__}')
print(f'GPU available      : {len(tf.config.list_physical_devices("GPU")) > 0}')


## Cell 3 — Mount Google Drive & Set Paths

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

# ── Base folders ──────────────────────────────────────────────────────────────
BASE_DIR    = '/content/drive/MyDrive/Internship_Datasets'
TASK6_DIR   = os.path.join(BASE_DIR, 'Task6_Nationality_Detection')
DATASET_DIR = os.path.join(TASK6_DIR, 'Datasets')
MODEL_DIR   = os.path.join(TASK6_DIR, 'Models')
PLOT_DIR    = os.path.join(TASK6_DIR, 'Plots')
OUTPUT_DIR  = os.path.join(TASK6_DIR, 'Outputs')

# ── Create all folders ────────────────────────────────────────────────────────
for d in [DATASET_DIR, MODEL_DIR, PLOT_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Colab working directory ───────────────────────────────────────────────────
WORK_DIR = '/content/task6_work'
os.makedirs(WORK_DIR, exist_ok=True)

print('Drive mounted. Folder structure created.')
print(f'Task6 folder: {TASK6_DIR}')

## Cell 4 — Upload kaggle.json & Configure Kaggle API

In [ ]:
# Upload your kaggle.json file when prompted
print('Please upload your kaggle.json file:')
uploaded = files.upload()

# Move kaggle.json to the correct location
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
shutil.move('kaggle.json', os.path.join(kaggle_dir, 'kaggle.json'))
os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)

print('kaggle.json configured successfully!')
!kaggle datasets list --search utkface | head -5

## Cell 5 — Download UTKFace Dataset (for Nationality + Age CNNs)

In [ ]:
UTKFACE_DIR = os.path.join(WORK_DIR, 'UTKFace')
UTKFACE_DRIVE = os.path.join(DATASET_DIR, 'UTKFace')  # Drive backup

if not os.path.exists(UTKFACE_DIR) or len(os.listdir(UTKFACE_DIR)) < 100:
    print('Downloading UTKFace dataset from Kaggle...')
    os.chdir(WORK_DIR)
    !kaggle datasets download -d jangedoo/utkface-new --unzip -p {WORK_DIR}/UTKFace_raw

    raw_dir = os.path.join(WORK_DIR, 'UTKFace_raw')
    os.makedirs(UTKFACE_DIR, exist_ok=True)
    for root, dirs, fnames in os.walk(raw_dir):
        for f in fnames:
            if f.endswith('.jpg') or f.endswith('.png'):
                shutil.copy(os.path.join(root, f), UTKFACE_DIR)

    print(f'UTKFace images collected: {len(os.listdir(UTKFACE_DIR))}')
else:
    print(f'UTKFace already in Colab: {len(os.listdir(UTKFACE_DIR))} images')

# ── Save sample to Drive (first 500 images as reference) ─────────────────────
if not os.path.exists(UTKFACE_DRIVE) or len(os.listdir(UTKFACE_DRIVE)) < 100:
    os.makedirs(UTKFACE_DRIVE, exist_ok=True)
    sample_imgs = os.listdir(UTKFACE_DIR)[:500]
    for f in sample_imgs:
        shutil.copy(os.path.join(UTKFACE_DIR, f), UTKFACE_DRIVE)
    print(f'✅ {len(sample_imgs)} UTKFace samples saved to Drive: {UTKFACE_DRIVE}')
else:
    print(f'Drive UTKFace already exists: {len(os.listdir(UTKFACE_DRIVE))} images')


## Cell 6 — Download FER2013 Dataset (for Emotion CNN)

In [ ]:
FER_DIR = os.path.join(WORK_DIR, 'FER2013')
FER_DRIVE = os.path.join(DATASET_DIR, 'FER2013')

if not os.path.exists(FER_DIR) or len(list(Path(FER_DIR).rglob('*.jpg')) + list(Path(FER_DIR).rglob('*.png'))) < 100:
    print('Downloading FER2013 dataset from Kaggle...')
    !kaggle datasets download -d msambare/fer2013 --unzip -p {FER_DIR}
    print('FER2013 downloaded.')
else:
    print('FER2013 already present in Colab.')

# Find the train folder
fer_train = None
for root, dirs, files_ in os.walk(FER_DIR):
    if 'train' in dirs:
        fer_train = os.path.join(root, 'train')
        break
    if os.path.basename(root) == 'train' and len(files_) > 0:
        fer_train = root
        break

print(f'FER2013 train folder: {fer_train}')
if fer_train:
    print('Emotion classes:', os.listdir(fer_train))

# ── Save folder structure to Drive (30 images per class as reference) ─────────
if fer_train and (not os.path.exists(FER_DRIVE) or len(list(Path(FER_DRIVE).rglob('*.jpg'))) < 50):
    os.makedirs(FER_DRIVE, exist_ok=True)
    saved = 0
    for cls_folder in os.listdir(fer_train):
        src_cls = os.path.join(fer_train, cls_folder)
        dst_cls = os.path.join(FER_DRIVE, cls_folder)
        os.makedirs(dst_cls, exist_ok=True)
        imgs = [f for f in os.listdir(src_cls) if f.endswith('.jpg') or f.endswith('.png')][:30]
        for img in imgs:
            shutil.copy(os.path.join(src_cls, img), dst_cls)
            saved += 1
    print(f'✅ {saved} FER2013 samples saved to Drive: {FER_DRIVE}')
else:
    print('Drive FER2013 already exists.')


## Cell 7 — Download Colour/Clothing Dataset (for Dress Colour CNN)

In [ ]:
# Using the Clothing Color Dataset from Kaggle
COLOUR_DIR = os.path.join(WORK_DIR, 'ClothingColour')

if not os.path.exists(COLOUR_DIR) or len(list(Path(COLOUR_DIR).rglob('*.jpg')) + list(Path(COLOUR_DIR).rglob('*.png'))) < 50:
    print('Downloading clothing colour dataset...')
    # Primary option: color classification dataset
    !kaggle datasets download -d biaiscience/clothes-color-classification --unzip -p {COLOUR_DIR} 2>/dev/null || \
     kaggle datasets download -d zhangjuefei/birds-bones-and-living-habits --unzip -p {COLOUR_DIR}_tmp 2>/dev/null || \
     echo 'Trying alternative dataset...'

    # Alternative: if above fails, try another colour dataset
    if len(list(Path(COLOUR_DIR).rglob('*.jpg'))) < 50:
        !kaggle datasets download -d olgabelitskaya/flower-color-images --unzip -p {COLOUR_DIR} 2>/dev/null || echo 'Using fallback synthetic colours'

    print(f'Colour dataset images: {len(list(Path(COLOUR_DIR).rglob("*.jpg")))}')
else:
    print(f'Colour dataset already present.')

# Check what we got
colour_images_count = len(list(Path(COLOUR_DIR).rglob('*.jpg'))) + len(list(Path(COLOUR_DIR).rglob('*.png')))
print(f'Total colour images available: {colour_images_count}')

## Cell 8 — Parse UTKFace Labels (Nationality + Age)

In [ ]:
# UTKFace filename format: [age]_[gender]_[race]_[date].jpg
# Race codes: 0=White(American), 1=Black(African), 2=Asian(Others), 3=Indian, 4=Others

NATIONALITY_MAP = {0: 'American', 1: 'African', 2: 'Others', 3: 'Indian', 4: 'Others'}
NATIONALITY_IDX = {'American': 0, 'African': 1, 'Indian': 2, 'Others': 3}
NATIONALITY_CLASSES = ['American', 'African', 'Indian', 'Others']

AGE_BUCKETS = {'Child': 0, 'Teen': 1, 'Adult': 2, 'Senior': 3}
def age_to_bucket(age):
    if age < 13:  return 'Child'
    elif age < 20: return 'Teen'
    elif age < 60: return 'Adult'
    else:          return 'Senior'

nat_data, age_data = [], []
parse_errors = 0

for fname in os.listdir(UTKFACE_DIR):
    if not (fname.endswith('.jpg') or fname.endswith('.png')):
        continue
    parts = fname.split('_')
    if len(parts) < 3:
        parse_errors += 1
        continue
    try:
        age  = int(parts[0])
        race = int(parts[2])
    except (ValueError, IndexError):
        parse_errors += 1
        continue
    if race not in NATIONALITY_MAP:
        parse_errors += 1
        continue

    fpath = os.path.join(UTKFACE_DIR, fname)
    nationality = NATIONALITY_MAP[race]
    nat_data.append((fpath, NATIONALITY_IDX[nationality]))
    age_data.append((fpath, AGE_BUCKETS[age_to_bucket(age)]))

print(f'Parsed {len(nat_data)} images ({parse_errors} skipped)')

# Distribution
nat_counts = {c: 0 for c in NATIONALITY_CLASSES}
for _, lbl in nat_data:
    nat_counts[NATIONALITY_CLASSES[lbl]] += 1
print('Nationality distribution:', nat_counts)

age_counts = {c: 0 for c in AGE_BUCKETS}
for _, lbl in age_data:
    for k, v in AGE_BUCKETS.items():
        if v == lbl: age_counts[k] += 1
print('Age distribution:', age_counts)

## Cell 9 — Visualise Sample UTKFace Images

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle('UTKFace Sample Images — Nationality Labels', fontsize=14, fontweight='bold')

# Show 2 samples per nationality class
for cls_idx, cls_name in enumerate(NATIONALITY_CLASSES):
    samples = [(p, l) for p, l in nat_data if l == cls_idx][:2]
    for j, (fpath, lbl) in enumerate(samples):
        ax = axes[j][cls_idx]
        img = cv2.imread(fpath)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(f'{cls_name}', fontweight='bold')
        ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'sample_utkface.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Sample visualisation saved.')

## Cell 10 — Helper: Load Images into Arrays

In [ ]:
def load_images(data_list, img_size, max_per_class=None, num_classes=4):
    """
    Load images from (filepath, label) list into numpy arrays.
    Args:
        data_list   : list of (filepath, int_label)
        img_size    : (H, W)
        max_per_class: cap per class to balance dataset
        num_classes : number of output classes
    Returns:
        X (N, H, W, 3), y (N, num_classes)
    """
    if max_per_class:
        from collections import defaultdict
        counts = defaultdict(int)
        filtered = []
        for path, lbl in data_list:
            if counts[lbl] < max_per_class:
                filtered.append((path, lbl))
                counts[lbl] += 1
        data_list = filtered

    X, y = [], []
    for fpath, lbl in data_list:
        img = cv2.imread(fpath)
        if img is None: continue
        img = cv2.resize(img, (img_size[1], img_size[0]))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        X.append(img.astype('float32') / 255.0)
        y.append(lbl)

    X = np.array(X)
    y = to_categorical(np.array(y), num_classes=num_classes)
    print(f'Loaded {len(X)} images, shape: {X.shape}')
    return X, y

print('Helper function defined.')

## Cell 11 — Helper: Build 4-Block CNN

In [ ]:
def build_cnn(input_shape, num_classes, name='CNN', filters=(32, 64, 128, 256)):
    """
    Custom 4-block CNN built from scratch.
    Architecture:
        Block 1-4: Conv2D → BatchNorm → ReLU → Conv2D → BatchNorm → ReLU → MaxPool → Dropout
        Head: GlobalAvgPool → Dense(256) → Dropout → Dense(num_classes, softmax)
    """
    model = models.Sequential(name=name)
    model.add(layers.Input(shape=input_shape))

    for i, f in enumerate(filters):
        model.add(layers.Conv2D(f, (3,3), padding='same', name=f'conv{i+1}a'))
        model.add(layers.BatchNormalization())
        model.add(layers.Activation('relu'))
        model.add(layers.Conv2D(f, (3,3), padding='same', name=f'conv{i+1}b'))
        model.add(layers.BatchNormalization())
        model.add(layers.Activation('relu'))
        model.add(layers.MaxPooling2D((2,2)))
        model.add(layers.Dropout(0.25))

    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.Dense(256, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(num_classes, activation='softmax'))

    return model

# Quick test
test_model = build_cnn((128, 128, 3), 4, name='TestCNN')
print(f'Test CNN params: {test_model.count_params():,}')
del test_model

## Cell 12 — Helper: Training Plots + Confusion Matrix

In [ ]:
def plot_training(history, model_name, save_dir):
    """Plot accuracy and loss curves for train/val."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f'{model_name} — Training Curves', fontsize=13, fontweight='bold')

    ax1.plot(history.history['accuracy'],     label='Train Acc',  color='steelblue')
    ax1.plot(history.history['val_accuracy'], label='Val Acc',    color='darkorange')
    ax1.set_title('Accuracy'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(alpha=0.3)

    ax2.plot(history.history['loss'],     label='Train Loss', color='steelblue')
    ax2.plot(history.history['val_loss'], label='Val Loss',   color='darkorange')
    ax2.set_title('Loss'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'{model_name}_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()


def plot_confusion(model, X_test, y_test, class_names, model_name, save_dir):
    """Plot confusion matrix and print classification report."""
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    y_true = np.argmax(y_test, axis=1)

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f'{model_name} — Confusion Matrix', fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'{model_name}_confusion.png'), dpi=150, bbox_inches='tight')
    plt.show()

    acc = np.mean(y_pred == y_true)
    print(f'\n{model_name} Test Accuracy: {acc:.4f}')
    print(classification_report(y_true, y_pred, target_names=class_names))
    return acc

print('Plot helper functions defined.')

## Cell 13 — Train Nationality CNN (UTKFace, 128×128)

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
print('Loading UTKFace for Nationality CNN...')
X_nat, y_nat = load_images(nat_data, img_size=(128, 128), max_per_class=1500, num_classes=4)

X_train_n, X_test_n, y_train_n, y_test_n = train_test_split(
    X_nat, y_nat, test_size=0.2, random_state=42, stratify=np.argmax(y_nat, axis=1))

# ── Data augmentation ─────────────────────────────────────────────────────────
aug_nat = ImageDataGenerator(
    rotation_range=15, width_shift_range=0.1, height_shift_range=0.1,
    horizontal_flip=True, zoom_range=0.1
)

# ── Build & compile model ─────────────────────────────────────────────────────
nat_model = build_cnn((128, 128, 3), num_classes=4, name='Nationality_CNN')
nat_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
nat_model.summary()

# ── Callbacks ─────────────────────────────────────────────────────────────────
nat_callbacks = [
    callbacks.EarlyStopping(patience=8, restore_best_weights=True, monitor='val_accuracy'),
    callbacks.ReduceLROnPlateau(factor=0.5, patience=4, monitor='val_loss', verbose=1)
]

# ── Train ─────────────────────────────────────────────────────────────────────
print('\nTraining Nationality CNN...')
nat_history = nat_model.fit(
    aug_nat.flow(X_train_n, y_train_n, batch_size=32),
    validation_data=(X_test_n, y_test_n),
    epochs=30,
    callbacks=nat_callbacks,
    verbose=1
)

## Cell 14 — Evaluate Nationality CNN

In [ ]:
plot_training(nat_history, 'Nationality_CNN', PLOT_DIR)
nat_acc = plot_confusion(nat_model, X_test_n, y_test_n, NATIONALITY_CLASSES, 'Nationality_CNN', PLOT_DIR)

# Save model
nat_model_path = os.path.join(MODEL_DIR, 'nationality_cnn.h5')
nat_model.save(nat_model_path)
print(f'\nNationality CNN saved to: {nat_model_path}')

## Cell 15 — Prepare FER2013 & Train Emotion CNN

In [ ]:
# FER2013 emotion classes
EMOTION_CLASSES = ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']
EMOTION_IDX = {c: i for i, c in enumerate(EMOTION_CLASSES)}

# ── Build emotion dataset from folder structure ────────────────────────────────
emo_data = []

# FER2013 folder names (may differ — handle common variants)
EMO_ALIAS = {
    'angry': 'Angry', 'disgust': 'Disgust', 'fear': 'Fear',
    'happy': 'Happy', 'neutral': 'Neutral', 'sad': 'Sad', 'surprise': 'Surprise'
}

for root, dirs, files_ in os.walk(FER_DIR):
    for fname in files_:
        if not (fname.endswith('.jpg') or fname.endswith('.png')): continue
        folder_name = os.path.basename(root).lower()
        if folder_name in EMO_ALIAS:
            emo_label = EMOTION_IDX[EMO_ALIAS[folder_name]]
            emo_data.append((os.path.join(root, fname), emo_label))

print(f'Emotion images found: {len(emo_data)}')

emo_counts = {c: 0 for c in EMOTION_CLASSES}
for _, lbl in emo_data:
    emo_counts[EMOTION_CLASSES[lbl]] += 1
print('Emotion distribution:', emo_counts)

## Cell 16 — Train Emotion CNN (FER2013, 64×64)

In [ ]:
print('Loading FER2013 for Emotion CNN...')
X_emo, y_emo = load_images(emo_data, img_size=(64, 64), max_per_class=1000, num_classes=7)

X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X_emo, y_emo, test_size=0.2, random_state=42, stratify=np.argmax(y_emo, axis=1))

aug_emo = ImageDataGenerator(
    rotation_range=10, width_shift_range=0.1,
    horizontal_flip=True, zoom_range=0.1
)

# 4-block CNN (smaller filters since 64×64 input)
emo_model = build_cnn((64, 64, 3), num_classes=7, name='Emotion_CNN', filters=(32, 64, 128, 128))
emo_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

emo_callbacks = [
    callbacks.EarlyStopping(patience=8, restore_best_weights=True, monitor='val_accuracy'),
    callbacks.ReduceLROnPlateau(factor=0.5, patience=4, monitor='val_loss', verbose=1)
]

print('Training Emotion CNN...')
emo_history = emo_model.fit(
    aug_emo.flow(X_train_e, y_train_e, batch_size=32),
    validation_data=(X_test_e, y_test_e),
    epochs=30,
    callbacks=emo_callbacks,
    verbose=1
)

## Cell 17 — Evaluate Emotion CNN

In [ ]:
plot_training(emo_history, 'Emotion_CNN', PLOT_DIR)
emo_acc = plot_confusion(emo_model, X_test_e, y_test_e, EMOTION_CLASSES, 'Emotion_CNN', PLOT_DIR)

emo_model_path = os.path.join(MODEL_DIR, 'emotion_cnn.h5')
emo_model.save(emo_model_path)
print(f'Emotion CNN saved to: {emo_model_path}')

## Cell 18 — Train Age CNN (UTKFace, 128×128)

In [ ]:
AGE_CLASSES = ['Child', 'Teen', 'Adult', 'Senior']

print('Loading UTKFace for Age CNN...')
X_age, y_age = load_images(age_data, img_size=(128, 128), max_per_class=1500, num_classes=4)

X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
    X_age, y_age, test_size=0.2, random_state=42, stratify=np.argmax(y_age, axis=1))

aug_age = ImageDataGenerator(
    rotation_range=15, width_shift_range=0.1,
    horizontal_flip=True, zoom_range=0.1
)

age_model = build_cnn((128, 128, 3), num_classes=4, name='Age_CNN')
age_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

age_callbacks = [
    callbacks.EarlyStopping(patience=8, restore_best_weights=True, monitor='val_accuracy'),
    callbacks.ReduceLROnPlateau(factor=0.5, patience=4, monitor='val_loss', verbose=1)
]

print('Training Age CNN...')
age_history = age_model.fit(
    aug_age.flow(X_train_a, y_train_a, batch_size=32),
    validation_data=(X_test_a, y_test_a),
    epochs=30,
    callbacks=age_callbacks,
    verbose=1
)

## Cell 19 — Evaluate Age CNN

In [ ]:
plot_training(age_history, 'Age_CNN', PLOT_DIR)
age_acc = plot_confusion(age_model, X_test_a, y_test_a, AGE_CLASSES, 'Age_CNN', PLOT_DIR)

age_model_path = os.path.join(MODEL_DIR, 'age_cnn.h5')
age_model.save(age_model_path)
print(f'Age CNN saved to: {age_model_path}')

## Cell 20 — Prepare Dress Colour Dataset

In [ ]:
# 8 Dress colour classes (matching Task 5 scheme)
COLOUR_CLASSES = ['Red', 'Blue', 'Green', 'Yellow', 'White', 'Black', 'Orange', 'Purple']
COLOUR_IDX = {c: i for i, c in enumerate(COLOUR_CLASSES)}

# ── Try to load real colour dataset ───────────────────────────────────────────
colour_data = []

COLOUR_ALIAS = {
    'red': 'Red', 'blue': 'Blue', 'green': 'Green', 'yellow': 'Yellow',
    'white': 'White', 'black': 'Black', 'orange': 'Orange', 'purple': 'Purple',
    'violet': 'Purple', 'grey': 'Black', 'gray': 'Black'
}

for root, dirs, files_ in os.walk(COLOUR_DIR):
    for fname in files_:
        if not (fname.endswith('.jpg') or fname.endswith('.png')): continue
        folder_name = os.path.basename(root).lower()
        if folder_name in COLOUR_ALIAS:
            cls_name = COLOUR_ALIAS[folder_name]
            if cls_name in COLOUR_IDX:
                colour_data.append((os.path.join(root, fname), COLOUR_IDX[cls_name]))

print(f'Real colour images found: {len(colour_data)}')

# ── Fallback: generate synthetic colour patches if dataset is too small ────────
SYNTHETIC_THRESHOLD = 200  # minimum images needed

if len(colour_data) < SYNTHETIC_THRESHOLD:
    print(f'Not enough real images ({len(colour_data)}). Generating synthetic colour patches...')

    synth_dir = os.path.join(WORK_DIR, 'SyntheticColours')
    os.makedirs(synth_dir, exist_ok=True)

    # BGR colour definitions (with variation)
    COLOUR_BGR = {
        'Red':    ([0,   0,   180], [30,  30,  255]),
        'Blue':   ([180, 0,   0],   [255, 30,  30]),
        'Green':  ([0,   140, 0],   [30,  220, 30]),
        'Yellow': ([0,   200, 200], [30,  255, 255]),
        'White':  ([200, 200, 200], [255, 255, 255]),
        'Black':  ([0,   0,   0],   [50,  50,  50]),
        'Orange': ([0,   100, 200], [30,  150, 255]),
        'Purple': ([120, 0,   100], [180, 30,  160]),
    }

    SAMPLES_PER_CLASS = 600
    colour_data = []

    for cls_name, (bgr_lo, bgr_hi) in COLOUR_BGR.items():
        cls_dir = os.path.join(synth_dir, cls_name)
        os.makedirs(cls_dir, exist_ok=True)
        for i in range(SAMPLES_PER_CLASS):
            patch = np.zeros((64, 64, 3), dtype=np.uint8)
            b = random.randint(bgr_lo[0], bgr_hi[0])
            g = random.randint(bgr_lo[1], bgr_hi[1])
            r = random.randint(bgr_lo[2], bgr_hi[2])
            patch[:] = [b, g, r]
            # Add subtle noise
            noise = np.random.randint(-15, 15, patch.shape, dtype=np.int16)
            patch = np.clip(patch.astype(np.int16) + noise, 0, 255).astype(np.uint8)
            fpath = os.path.join(cls_dir, f'{cls_name}_{i:04d}.jpg')
            cv2.imwrite(fpath, patch)
            colour_data.append((fpath, COLOUR_IDX[cls_name]))

    print(f'Generated {len(colour_data)} synthetic colour patches.')

# Class distribution
c_counts = {c: 0 for c in COLOUR_CLASSES}
for _, lbl in colour_data:
    c_counts[COLOUR_CLASSES[lbl]] += 1
print('Colour distribution:', c_counts)

## Cell 21 — Train Dress Colour CNN (64×64)

In [ ]:
print('Loading dress colour data...')
X_col, y_col = load_images(colour_data, img_size=(64, 64), max_per_class=500, num_classes=8)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_col, y_col, test_size=0.2, random_state=42, stratify=np.argmax(y_col, axis=1))

aug_col = ImageDataGenerator(
    rotation_range=20, width_shift_range=0.15, height_shift_range=0.15,
    horizontal_flip=True, zoom_range=0.15, brightness_range=[0.8, 1.2]
)

# 3-block CNN for colour (like Task 5)
colour_model = build_cnn((64, 64, 3), num_classes=8, name='Colour_CNN', filters=(32, 64, 128, 128))
colour_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

col_callbacks = [
    callbacks.EarlyStopping(patience=8, restore_best_weights=True, monitor='val_accuracy'),
    callbacks.ReduceLROnPlateau(factor=0.5, patience=4, monitor='val_loss', verbose=1)
]

print('Training Dress Colour CNN...')
col_history = colour_model.fit(
    aug_col.flow(X_train_c, y_train_c, batch_size=32),
    validation_data=(X_test_c, y_test_c),
    epochs=30,
    callbacks=col_callbacks,
    verbose=1
)

## Cell 22 — Evaluate Dress Colour CNN

In [ ]:
plot_training(col_history, 'Colour_CNN', PLOT_DIR)
col_acc = plot_confusion(colour_model, X_test_c, y_test_c, COLOUR_CLASSES, 'Colour_CNN', PLOT_DIR)

colour_model_path = os.path.join(MODEL_DIR, 'dress_colour_cnn.h5')
colour_model.save(colour_model_path)
print(f'Dress Colour CNN saved to: {colour_model_path}')

## Cell 23 — Full Inference Pipeline (Conditional Branching)

In [ ]:
def predict_nationality(img_bgr):
    """Predict nationality from a BGR face image."""
    img = cv2.resize(img_bgr, (128, 128))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype('float32') / 255.0
    pred = nat_model.predict(img[np.newaxis], verbose=0)[0]
    cls  = NATIONALITY_CLASSES[np.argmax(pred)]
    conf = float(np.max(pred))
    return cls, conf

def predict_emotion(img_bgr):
    """Predict emotion from a BGR face image."""
    img = cv2.resize(img_bgr, (64, 64))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype('float32') / 255.0
    pred = emo_model.predict(img[np.newaxis], verbose=0)[0]
    cls  = EMOTION_CLASSES[np.argmax(pred)]
    conf = float(np.max(pred))
    return cls, conf

def predict_age(img_bgr):
    """Predict age group from a BGR face image."""
    img = cv2.resize(img_bgr, (128, 128))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype('float32') / 255.0
    pred = age_model.predict(img[np.newaxis], verbose=0)[0]
    cls  = AGE_CLASSES[np.argmax(pred)]
    conf = float(np.max(pred))
    return cls, conf

def predict_colour(img_bgr):
    """Predict dress colour from a BGR image (torso region or full image)."""
    img = cv2.resize(img_bgr, (64, 64))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype('float32') / 255.0
    pred = colour_model.predict(img[np.newaxis], verbose=0)[0]
    cls  = COLOUR_CLASSES[np.argmax(pred)]
    conf = float(np.max(pred))
    return cls, conf


def full_pipeline(image_path):
    """
    Full Task 6 inference pipeline.
    Returns a results dict following the task specification:
        Indian   → Nationality + Emotion + Age + Dress Colour
        American → Nationality + Emotion + Age
        African  → Nationality + Emotion + Dress Colour
        Others   → Nationality + Emotion
    """
    img = cv2.imread(image_path)
    if img is None:
        return {'error': f'Cannot read image: {image_path}'}

    results = {}

    # Step 1 — Nationality (always runs)
    nationality, nat_conf = predict_nationality(img)
    results['Nationality'] = {'label': nationality, 'confidence': f'{nat_conf:.2%}'}

    # Step 2 — Emotion (always runs for all nationalities)
    emotion, emo_conf = predict_emotion(img)
    results['Emotion'] = {'label': emotion, 'confidence': f'{emo_conf:.2%}'}

    # Step 3 — Age (Indian or American only)
    if nationality in ('Indian', 'American'):
        age, age_conf = predict_age(img)
        results['Age'] = {'label': age, 'confidence': f'{age_conf:.2%}'}

    # Step 4 — Dress Colour (Indian or African only)
    if nationality in ('Indian', 'African'):
        colour, col_conf = predict_colour(img)
        results['Dress Colour'] = {'label': colour, 'confidence': f'{col_conf:.2%}'}

    return results


print('Inference pipeline defined.')
print('Active models:', [m.name for m in [nat_model, emo_model, age_model, colour_model]])

## Cell 24 — Test Inference on Sample Images

In [ ]:
# Pick 8 random test images from UTKFace and run the full pipeline
test_samples = random.sample([p for p, _ in nat_data], min(8, len(nat_data)))

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle('Task 6 — Full Pipeline Inference on Test Images', fontsize=14, fontweight='bold')

for idx, (ax, fpath) in enumerate(zip(axes.flat, test_samples)):
    results = full_pipeline(fpath)
    img = cv2.imread(fpath)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    ax.imshow(img)
    ax.axis('off')

    # Build label text
    lines = []
    for key, val in results.items():
        if isinstance(val, dict):
            lines.append(f"{key}: {val['label']} ({val['confidence']})")
        else:
            lines.append(f'{key}: {val}')

    ax.set_title('\n'.join(lines), fontsize=7, ha='left', x=0)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'pipeline_test_results.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Pipeline test saved to Outputs/')

## Cell 25 — Model Performance Summary Chart

In [ ]:
model_names  = ['Nationality CNN', 'Emotion CNN', 'Age CNN', 'Dress Colour CNN']
model_accs   = [nat_acc, emo_acc, age_acc, col_acc]
bar_colours  = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(model_names, [a * 100 for a in model_accs], color=bar_colours, edgecolor='white', linewidth=1.2)

for bar, acc in zip(bars, model_accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{acc:.2%}', ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_ylim(0, 110)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Task 6 — All Model Accuracies', fontsize=14, fontweight='bold')
ax.axhline(y=70, color='red', linestyle='--', alpha=0.5, label='70% baseline')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'all_model_accuracies.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n===== FINAL SUMMARY =====')
for name, acc in zip(model_names, model_accs):
    print(f'  {name:<22}: {acc:.4f} ({acc:.2%})')

## Cell 26 — Save GUI Script to Drive

In [ ]:
GUI_CODE = '''
#!/usr/bin/env python3
"""
Task 6 — Nationality Detection GUI
Run this script on your local machine (not in Colab).
Requirements: pip install tensorflow opencv-python PyQt5

Usage:
    python task6_gui.py

Set MODEL_DIR to the folder where your .h5 files are saved.
"""

import sys, os
import numpy as np
import cv2
from PyQt5.QtWidgets import (
    QApplication, QMainWindow, QWidget, QVBoxLayout, QHBoxLayout,
    QPushButton, QLabel, QFileDialog, QFrame, QScrollArea, QSizePolicy
)
from PyQt5.QtGui import QPixmap, QImage, QFont, QColor, QPalette
from PyQt5.QtCore import Qt, QThread, pyqtSignal
import tensorflow as tf

# ── CONFIG — update this path ──────────────────────────────────────────────────
MODEL_DIR = r"C:/InternshipModels/Task6_Nationality_Detection/Models"
# ──────────────────────────────────────────────────────────────────────────────

NATIONALITY_CLASSES = ["American", "African", "Indian", "Others"]
EMOTION_CLASSES     = ["Angry", "Disgust", "Fear", "Happy", "Neutral", "Sad", "Surprise"]
AGE_CLASSES         = ["Child", "Teen", "Adult", "Senior"]
COLOUR_CLASSES      = ["Red", "Blue", "Green", "Yellow", "White", "Black", "Orange", "Purple"]

NATIONALITY_FLAG = {
    "Indian": "[IN]", "American": "[US]", "African": "[AF]", "Others": "[--]"
}


def load_models():
    models = {}
    for name, fname in [
        ("nationality", "nationality_cnn.h5"),
        ("emotion",     "emotion_cnn.h5"),
        ("age",         "age_cnn.h5"),
        ("colour",      "dress_colour_cnn.h5"),
    ]:
        path = os.path.join(MODEL_DIR, fname)
        if os.path.exists(path):
            models[name] = tf.keras.models.load_model(path)
            print(f"Loaded: {fname}")
        else:
            print(f"WARNING: {fname} not found at {path}")
    return models


def preprocess(img_bgr, size):
    img = cv2.resize(img_bgr, size)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype("float32") / 255.0
    return img[np.newaxis]


def run_pipeline(img_bgr, models):
    results = []

    # Nationality
    pred = models["nationality"].predict(preprocess(img_bgr, (128,128)), verbose=0)[0]
    nat  = NATIONALITY_CLASSES[np.argmax(pred)]
    results.append(("Nationality", nat, float(np.max(pred))))

    # Emotion (always)
    pred = models["emotion"].predict(preprocess(img_bgr, (64,64)), verbose=0)[0]
    results.append(("Emotion", EMOTION_CLASSES[np.argmax(pred)], float(np.max(pred))))

    # Age — Indian or American only
    if nat in ("Indian", "American") and "age" in models:
        pred = models["age"].predict(preprocess(img_bgr, (128,128)), verbose=0)[0]
        results.append(("Age", AGE_CLASSES[np.argmax(pred)], float(np.max(pred))))

    # Dress Colour — Indian or African only
    if nat in ("Indian", "African") and "colour" in models:
        pred = models["colour"].predict(preprocess(img_bgr, (64,64)), verbose=0)[0]
        results.append(("Dress Colour", COLOUR_CLASSES[np.argmax(pred)], float(np.max(pred))))

    return results, nat


class InferenceThread(QThread):
    finished = pyqtSignal(list, str)
    error    = pyqtSignal(str)

    def __init__(self, image_path, models):
        super().__init__()
        self.image_path = image_path
        self.models = models

    def run(self):
        try:
            img = cv2.imread(self.image_path)
            if img is None:
                self.error.emit("Cannot read image.")
                return
            results, nationality = run_pipeline(img, self.models)
            self.finished.emit(results, nationality)
        except Exception as e:
            self.error.emit(str(e))


class MainWindow(QMainWindow):
    def __init__(self, models):
        super().__init__()
        self.models = models
        self.setWindowTitle("Task 6 — Nationality Detection")
        self.setMinimumSize(860, 600)
        self._build_ui()

    def _build_ui(self):
        central = QWidget()
        self.setCentralWidget(central)
        root = QVBoxLayout(central)
        root.setSpacing(12)
        root.setContentsMargins(16, 16, 16, 16)

        # Title
        title = QLabel("Nationality Detection System")
        title.setFont(QFont("Arial", 18, QFont.Bold))
        title.setAlignment(Qt.AlignCenter)
        root.addWidget(title)

        subtitle = QLabel("Upload an image to detect Nationality, Emotion, Age, and Dress Colour")
        subtitle.setAlignment(Qt.AlignCenter)
        subtitle.setStyleSheet("color: #666; font-size: 11px;")
        root.addWidget(subtitle)

        # Content row: image | results
        content = QHBoxLayout()
        root.addLayout(content)

        # Image panel
        img_frame = QFrame()
        img_frame.setFrameShape(QFrame.StyledPanel)
        img_frame.setStyleSheet("background:#1a1a2e; border-radius:8px;")
        img_layout = QVBoxLayout(img_frame)
        self.img_label = QLabel("No image loaded")
        self.img_label.setAlignment(Qt.AlignCenter)
        self.img_label.setMinimumSize(380, 380)
        self.img_label.setStyleSheet("color:#aaa; font-size:13px;")
        img_layout.addWidget(self.img_label)
        content.addWidget(img_frame, 55)

        # Results panel
        res_frame = QFrame()
        res_frame.setFrameShape(QFrame.StyledPanel)
        res_frame.setStyleSheet("background:#0f3460; border-radius:8px;")
        res_layout = QVBoxLayout(res_frame)

        res_title = QLabel("Results")
        res_title.setFont(QFont("Arial", 13, QFont.Bold))
        res_title.setStyleSheet("color:white;")
        res_title.setAlignment(Qt.AlignCenter)
        res_layout.addWidget(res_title)

        scroll = QScrollArea()
        scroll.setWidgetResizable(True)
        scroll.setStyleSheet("border:none;")
        self.results_container = QWidget()
        self.results_layout = QVBoxLayout(self.results_container)
        self.results_layout.setAlignment(Qt.AlignTop)
        scroll.setWidget(self.results_container)
        res_layout.addWidget(scroll)

        self._add_placeholder_result()
        content.addWidget(res_frame, 45)

        # Buttons
        btn_row = QHBoxLayout()
        self.upload_btn = QPushButton("Upload Image")
        self.upload_btn.setMinimumHeight(42)
        self.upload_btn.setStyleSheet(
            "QPushButton{background:#16213e;color:white;border-radius:6px;font-size:13px;font-weight:bold;}"
            "QPushButton:hover{background:#0f3460;}"
        )
        self.upload_btn.clicked.connect(self._open_file)

        self.analyze_btn = QPushButton("Analyze Image")
        self.analyze_btn.setMinimumHeight(42)
        self.analyze_btn.setEnabled(False)
        self.analyze_btn.setStyleSheet(
            "QPushButton{background:#e94560;color:white;border-radius:6px;font-size:13px;font-weight:bold;}"
            "QPushButton:hover{background:#c73652;}"
            "QPushButton:disabled{background:#555;}"
        )
        self.analyze_btn.clicked.connect(self._analyze)

        btn_row.addWidget(self.upload_btn)
        btn_row.addWidget(self.analyze_btn)
        root.addLayout(btn_row)

        self.status_label = QLabel("Ready")
        self.status_label.setAlignment(Qt.AlignCenter)
        self.status_label.setStyleSheet("color:#888; font-size:10px;")
        root.addWidget(self.status_label)

        self.setStyleSheet("background:#16213e; color:white;")

    def _add_placeholder_result(self):
        lbl = QLabel("Upload an image to see results here.")
        lbl.setStyleSheet("color:#aaa; font-style:italic;")
        lbl.setAlignment(Qt.AlignCenter)
        self.results_layout.addWidget(lbl)

    def _open_file(self):
        path, _ = QFileDialog.getOpenFileName(
            self, "Open Image", "",
            "Images (*.jpg *.jpeg *.png *.bmp *.webp)"
        )
        if not path: return
        self.current_path = path
        pixmap = QPixmap(path).scaled(370, 370, Qt.KeepAspectRatio, Qt.SmoothTransformation)
        self.img_label.setPixmap(pixmap)
        self.analyze_btn.setEnabled(True)
        self.status_label.setText(f"Image loaded: {os.path.basename(path)}")

    def _analyze(self):
        self.analyze_btn.setEnabled(False)
        self.status_label.setText("Analyzing...")
        self._clear_results()

        self.thread = InferenceThread(self.current_path, self.models)
        self.thread.finished.connect(self._show_results)
        self.thread.error.connect(self._show_error)
        self.thread.start()

    def _clear_results(self):
        while self.results_layout.count():
            item = self.results_layout.takeAt(0)
            if item.widget(): item.widget().deleteLater()

    def _show_results(self, results, nationality):
        flag = NATIONALITY_FLAG.get(nationality, "[--]")

        COLOURS = {
            "Nationality":   "#2196F3",
            "Emotion":       "#4CAF50",
            "Age":           "#FF9800",
            "Dress Colour":  "#9C27B0",
        }

        for label_name, value, confidence in results:
            frame = QFrame()
            frame.setStyleSheet(f"background:#1a1a3e; border-radius:6px; margin:2px;")
            layout = QVBoxLayout(frame)
            layout.setContentsMargins(10, 8, 10, 8)

            display = f"{flag} {value}" if label_name == "Nationality" else value
            colour  = COLOURS.get(label_name, "white")

            top = QLabel(f"<b style='color:{colour};font-size:11px;'>{label_name}</b>")
            top.setTextFormat(Qt.RichText)

            val_lbl = QLabel(f"<span style='font-size:20px; font-weight:bold;'>{display}</span>")
            val_lbl.setTextFormat(Qt.RichText)

            conf_lbl = QLabel(f"Confidence: {confidence:.2%}")
            conf_lbl.setStyleSheet("color:#aaa; font-size:10px;")

            layout.addWidget(top)
            layout.addWidget(val_lbl)
            layout.addWidget(conf_lbl)
            self.results_layout.addWidget(frame)

        self.analyze_btn.setEnabled(True)
        self.status_label.setText(f"Analysis complete — {nationality} pipeline applied.")

    def _show_error(self, msg):
        lbl = QLabel(f"Error: {msg}")
        lbl.setStyleSheet("color:red;")
        self.results_layout.addWidget(lbl)
        self.analyze_btn.setEnabled(True)
        self.status_label.setText("Error during analysis.")


if __name__ == "__main__":
    app = QApplication(sys.argv)
    app.setStyle("Fusion")
    print("Loading models...")
    models = load_models()
    window = MainWindow(models)
    window.show()
    sys.exit(app.exec_())
'''

# Save GUI script to Drive
gui_path = os.path.join(TASK6_DIR, 'task6_gui.py')
with open(gui_path, 'w') as f:
    f.write(GUI_CODE.strip())

print(f'GUI script saved to: {gui_path}')
print()
print('To run the GUI:')
print('  1. Download task6_gui.py from Google Drive')
print('  2. Update MODEL_DIR to point to your saved .h5 files')
print('  3. Run: python task6_gui.py')

## Cell 27 — Final Summary & Drive File Tree

## Cell 28 — 🧪 Test on Your Own Image

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 28 — Test on Your Own Image                              ║
# ║  Upload any face image and the pipeline will predict:          ║
# ║    • Nationality  (always)                                     ║
# ║    • Emotion      (always)                                     ║
# ║    • Age          (Indian / American only)                     ║
# ║    • Dress Colour (Indian / African only)                      ║
# ╚══════════════════════════════════════════════════════════════════╝

from google.colab import files as colab_files
import IPython.display as ipd

print('Please upload an image file (jpg / png):')
uploaded = colab_files.upload()

if not uploaded:
    print('No file uploaded.')
else:
    img_filename = list(uploaded.keys())[0]
    img_path = f'/content/{img_filename}'

    # ── Run full pipeline ─────────────────────────────────────────────
    results = full_pipeline(img_path)

    # ── Display image + results side by side ──────────────────────────
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    fig, (ax_img, ax_res) = plt.subplots(1, 2, figsize=(12, 5),
                                          gridspec_kw={'width_ratios': [1, 1]})
    fig.patch.set_facecolor('#1a1a2e')

    # Left: image
    ax_img.imshow(img_rgb)
    ax_img.axis('off')
    ax_img.set_title(img_filename, color='white', fontsize=10, pad=8)

    # Right: results panel
    ax_res.set_facecolor('#0f3460')
    ax_res.axis('off')

    LABEL_COLOURS = {
        'Nationality':  '#2196F3',
        'Emotion':      '#4CAF50',
        'Age':          '#FF9800',
        'Dress Colour': '#CE93D8',
    }
    FLAG = {'Indian': '🇮🇳', 'American': '🇺🇸', 'African': '🌍', 'Others': '🌐'}

    y_pos = 0.92
    ax_res.text(0.5, 0.98, 'RESULTS', color='white', fontsize=14,
                fontweight='bold', ha='center', va='top',
                transform=ax_res.transAxes)

    for key, val in results.items():
        if isinstance(val, dict):
            label_val  = val['label']
            confidence = val['confidence']
        else:
            label_val  = str(val)
            confidence = ''

        icon = FLAG.get(label_val, '') if key == 'Nationality' else ''
        colour = LABEL_COLOURS.get(key, 'white')

        ax_res.text(0.08, y_pos, key.upper(), color=colour, fontsize=9,
                    fontweight='bold', transform=ax_res.transAxes, va='top')
        ax_res.text(0.08, y_pos - 0.07,
                    f'{icon} {label_val}  ({confidence})',
                    color='white', fontsize=13, fontweight='bold',
                    transform=ax_res.transAxes, va='top')
        y_pos -= 0.22

    # Pipeline note
    nat = results.get('Nationality', {}).get('label', '')
    pipeline_note = {
        'Indian':   'Pipeline: Nationality + Emotion + Age + Dress Colour',
        'American': 'Pipeline: Nationality + Emotion + Age',
        'African':  'Pipeline: Nationality + Emotion + Dress Colour',
        'Others':   'Pipeline: Nationality + Emotion only',
    }.get(nat, '')
    ax_res.text(0.08, 0.04, pipeline_note, color='#aaaaaa', fontsize=8,
                transform=ax_res.transAxes, va='bottom', style='italic')

    plt.tight_layout()
    out_path = os.path.join(OUTPUT_DIR, f'test_{img_filename}')
    plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
    plt.show()

    print(f'\n✅ Result saved to Drive: {out_path}')
    print()
    print('=' * 40)
    for key, val in results.items():
        if isinstance(val, dict):
            print(f'  {key:<15}: {val["label"]}  ({val["confidence"]})')
    print('=' * 40)


In [ ]:
print('='*55)
print(' TASK 6 — NATIONALITY DETECTION — COMPLETE')
print('='*55)

print('\n Models saved to Drive:')
for fname in os.listdir(MODEL_DIR):
    fpath = os.path.join(MODEL_DIR, fname)
    size_mb = os.path.getsize(fpath) / (1024*1024)
    print(f'   {fname:<30} {size_mb:.1f} MB')

print('\n Plots saved to Drive:')
for fname in os.listdir(PLOT_DIR):
    print(f'   {fname}')

print('\n Outputs saved to Drive:')
for fname in os.listdir(OUTPUT_DIR):
    print(f'   {fname}')

print('\n Pipeline rules applied:')
print('   Indian   → Nationality + Emotion + Age + Dress Colour')
print('   American → Nationality + Emotion + Age')
print('   African  → Nationality + Emotion + Dress Colour')
print('   Others   → Nationality + Emotion')

print(f'\n Test Accuracies:')
for name, acc in zip(model_names, model_accs):
    bar = '|' * int(acc * 30)
    print(f'   {name:<22}: {bar:<30} {acc:.2%}')

print('\n Drive folder: Internship_Datasets/Task6_Nationality_Detection/')
print('='*55)